In [28]:
import json
import uuid
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import udahub

# Udahub Application

## Core Database

**Init DB**

In [29]:
udahub_db = "data/core/udahub.db"

In [30]:
reset_db(udahub_db)

✅ Removed existing data/core/udahub.db
2026-04-27 09:38:17,009 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-27 09:38:17,009 INFO sqlalchemy.engine.Engine COMMIT
✅ Recreated data/core/udahub.db with fresh schema


In [31]:
engine = create_engine(f"sqlite:///{udahub_db}", echo=False)
udahub.Base.metadata.create_all(bind=engine)

**Account**

In [19]:
account_id = "cultpass"
account_name = "CultPass Card"

In [32]:
print(len(cultpass_articles))

4


In [33]:
with get_session(engine) as session:
    account = udahub.Account(
        account_id=account_id,
        account_name=account_name,
    )
    session.add(account)

## Integrations

**Knowledge Base**

In [ ]:
# TODO: Create additional 10 articles

# created additional 10 articles in cultpass_articles.jsonl


In [35]:
cultpass_articles = []

with open('data/external/cultpass_articles.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_articles.append(json.loads(line))

In [36]:
cultpass_articles

[{'title': 'How to Reserve a Spot for an Event',
  'content': 'If a user asks how to reserve an event:\n\n- Guide them to the CultPass app\n- Instruct them to browse the experience catalog and tap \'Reserve\'\n- If it\'s a premium or limited event, check if reservation confirmation is required via email\n- Remind them to arrive at least 15 minutes early with their QR code visible\n\n**Suggested phrasing:**\n"You can reserve an experience by opening the CultPass app, selecting your desired event, and tapping \'Reserve\'. Be sure to arrive 15 minutes early with your QR code ready."',
  'tags': 'reservation, events, booking, attendance'},
 {'title': "What's Included in a CultPass Subscription",
  'content': 'Each user is entitled to 4 cultural experiences per month, which may include:\n- Art exhibitions\n- Museum entries\n- Music concerts\n- Film screenings and more\n\nSome premium experiences may require an additional fee (visible in the app).\n\n**Suggested phrasing:**\n"Your CultPass s

In [37]:
if len(cultpass_articles) < 14:
    raise AssertionError("You should load the articles with at least 14 records")

In [38]:
with get_session(engine) as session:
    kb = []
    for article in cultpass_articles:
        knowledge = udahub.Knowledge(
            article_id=str(uuid.uuid4()),
            account_id=account_id,
            title=article["title"],
            content=article["content"],
            tags=article["tags"]
        )
        kb.append(knowledge)
    session.add_all(kb) 
    

**Ticket**

In [39]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [40]:
ticket_info = {
    "status": "open",
    "content": "I can't log in to my Cultpass account.",
    "owner_id": cultpass_users[0]["id"],
    "owner_name": cultpass_users[0]["name"],
    "role": "user",
    "channel": "chat",
    "tags": "login, access",
}

In [42]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()

    if not user:
        user = udahub.User(
            user_id=str(uuid.uuid4()),
            account_id=account_id,
            external_user_id=ticket_info["owner_id"],
            user_name=ticket_info["owner_name"],
        )
    
    ticket = udahub.Ticket(
        ticket_id=str(uuid.uuid4()),
        account_id=account_id,
        user_id=user.user_id,
        channel=ticket_info["channel"],
    )
    metadata = udahub.TicketMetadata(
        ticket_id=ticket.ticket_id,
        status=ticket_info["status"],
        main_issue_type=None,
        tags=ticket_info["tags"],
    )

    first_message = udahub.TicketMessage(
        message_id=str(uuid.uuid4()),
        ticket_id=ticket.ticket_id,
        role=ticket_info["role"],
        content=ticket_info["content"],
    )

    session.add_all([user, ticket, metadata, first_message])


# Tests

In [43]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    print(account)

<Account(account_id='cultpass', account_name='CultPass Card')>


In [44]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    for article in account.knowledge_articles:
        print(article)

<Knowledge(article_id='d0be1d25-d24c-439c-b6de-08449ce319ba', title='How to Reserve a Spot for an Event')>
<Knowledge(article_id='c7c6580b-118b-4bdd-a6c4-8a32f39a1702', title='What's Included in a CultPass Subscription')>
<Knowledge(article_id='65e2c116-3f76-42c7-a94e-eb4219c35dfc', title='How to Cancel or Pause a Subscription')>
<Knowledge(article_id='64bfcbf5-d18e-43f4-b223-f25b0340f0d6', title='How to Handle Login Issues?')>
<Knowledge(article_id='d01b7140-67b0-4dd2-83e7-85eb1ff7300c', title='Understanding Membership Tiers')>
<Knowledge(article_id='66854f37-1caf-4dc6-a89b-45b53a60b106', title='What to Do If an Event Is Fully Booked')>
<Knowledge(article_id='c5b243e3-c7e3-477a-aa38-fbae9f8a42c5', title='How to Check Your Monthly Quota')>
<Knowledge(article_id='07d2ab92-e1fa-443e-a20f-2d24477580a1', title='Late Arrival Policy for Events')>
<Knowledge(article_id='21a440ad-6078-4e9d-aaed-1b9875d834db', title='How QR Code Entry Works')>
<Knowledge(article_id='849c7eeb-1354-45cd-865b-20f5

In [46]:
with get_session(engine) as session:
    users = session.query(udahub.User).all()
    for user in users:
        print(user)

<User(user_id='08d3f947-9995-4182-aa0c-f5bc6602684e', user_name='Alice Kingsley', external_user_id='a4ab87')>


In [47]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()
    
    ticket:udahub.Ticket = user.tickets[0]
    for message in ticket.messages:
        print(message)

<TicketMessage(message_id='bbf1de7a-a21e-4891-afbd-efea91738371', role='user', content='I can't log in to my Cultpass ...')>
